# LSTM training: cross-city weather-transfer and operational energy models

This notebook uses the existing CSV candidate tables—**no feature rebuilding**. It forms a 24-hour sequence ending at each prediction timestamp and trains two LSTM variants:

- **Transfer/weather LSTM:** calendar + current/forecast weather + historical weather; never receives load-history features.
- **Operational LSTM:** exactly the same sequence features plus `load_lag_1h`, `load_lag_24h`, and `load_lag_168h`.

Each scaler/imputer is fitted only on the relevant training sequence rows. Windows never cross an identified timestamp gap.

In [1]:
from pathlib import Path
import json
import gc
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd().parent
DATA_DIR = BASE / 'processed_feature_candidates_cross_city_v1'
RESULTS_DIR = BASE / 'lstm_training_results_cross_city_v1'
RESULTS_DIR.mkdir(exist_ok=True)
TARGET = 'hourly_load_mw'
LOOKBACK_HOURS = 24
BATCH_SIZE = 256
EPOCHS = 60
RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
print('TensorFlow:', tf.__version__)
print('Feature CSV directory:', DATA_DIR)

TensorFlow: 2.10.0
Feature CSV directory: D:\Downloads\BTP\model_training\lstm_training_3\processed_feature_candidates_cross_city_v1


## Feature connection

These are the compact, necessary starting features selected from the candidate CSVs. All weather-lag columns were built without crossing the two 25-hour gaps. The LSTM additionally sees a contiguous 24-hour history of these rows, letting it learn thermal persistence directly.

In [2]:
CALENDAR_FEATURES = [
    'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month', 'sin_doy', 'cos_doy',
    'is_weekend', 'is_holiday_us_federal', 'is_business_hour', 'is_night',
]
CURRENT_WEATHER_FEATURES = [
    'temperature_c', 'dew_point_c', 'station_pressure_kpa', 'wind_speed_ms',
    'wind_direction_sin', 'wind_direction_cos',
    'extraterrestrial_direct_normal_radiation_wh_m2', 'broadband_aerosol_optical_depth',
]
CROSS_VARIABLE_FEATURES = [
    'temperature_dewpoint_spread_c', 'humidex_c',
    'cooling_degree_base_18c', 'heating_degree_base_18c',
    'cooling_degree_base_22c', 'heating_degree_base_22c',
    'temperature_x_sin_hour', 'temperature_x_cos_hour',
    'humidex_x_sin_hour', 'humidex_x_cos_hour',
]
WEATHER_HISTORY_FEATURES = [
    'temperature_c_lag_1h', 'temperature_c_lag_3h', 'temperature_c_lag_6h',
    'temperature_c_lag_12h', 'temperature_c_lag_24h',
    'temperature_c_prior_6h_mean', 'temperature_c_prior_24h_mean',
    'temperature_c_prior_6h_max', 'temperature_c_prior_24h_max',
    'temperature_change_1h', 'temperature_change_6h', 'temperature_change_24h',
    'dew_point_c_lag_3h', 'dew_point_c_lag_6h', 'dew_point_c_lag_24h',
    'dew_point_c_prior_6h_mean', 'dew_point_c_prior_24h_mean',
    'wind_speed_ms_lag_3h', 'wind_speed_ms_lag_24h',
    'wind_speed_ms_prior_6h_mean', 'wind_speed_ms_prior_24h_mean',
]
TRANSFER_FEATURES = CALENDAR_FEATURES + CURRENT_WEATHER_FEATURES + CROSS_VARIABLE_FEATURES + WEATHER_HISTORY_FEATURES
OPERATIONAL_LOAD_FEATURES = ['load_lag_1h', 'load_lag_24h', 'load_lag_168h']
OPERATIONAL_FEATURES = TRANSFER_FEATURES + OPERATIONAL_LOAD_FEATURES
print('Transfer sequence width:', len(TRANSFER_FEATURES))
print('Operational sequence width:', len(OPERATIONAL_FEATURES))

Transfer sequence width: 51
Operational sequence width: 54


In [3]:
def load_city(city):
    path = DATA_DIR / f'{city}_feature_candidates.csv'
    df = pd.read_csv(path, parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    required = set(OPERATIONAL_FEATURES + [TARGET, '_segment_id', 'city', 'timestamp'])
    missing = sorted(required - set(df.columns))
    assert not missing, f'{city} missing required columns: {missing}'
    return df

cities = {city: load_city(city) for city in ('las_vegas', 'phoenix')}
for city, df in cities.items():
    print(city, df.shape, 'segments:', df['_segment_id'].nunique())

las_vegas (70080, 233) segments: 3
phoenix (70080, 233) segments: 3


## Sequence, preprocessing, and LSTM helpers

A target at time *t* uses rows `t−23` through `t`. The weather at *t* must be the observation available at prediction time, or the forecast for *t* in a day-ahead deployment. The operational load lags are strictly earlier than *t*.

In [4]:
def make_sequences(df, features, lookback=LOOKBACK_HOURS):
    xs, ys, timestamps, cities_out = [], [], [], []
    for _, part in df.groupby('_segment_id', sort=False):
        if len(part) < lookback:
            continue
        values = part[features].to_numpy(dtype=np.float32)
        # Sliding view is ordered (samples, features, lookback); transpose to LSTM order.
        windows = np.lib.stride_tricks.sliding_window_view(values, lookback, axis=0).transpose(0, 2, 1).copy()
        xs.append(windows)
        ys.append(part[TARGET].to_numpy(dtype=np.float32)[lookback - 1:])
        timestamps.append(part['timestamp'].to_numpy()[lookback - 1:])
        cities_out.append(part['city'].to_numpy()[lookback - 1:])
    return (np.concatenate(xs), np.concatenate(ys), np.concatenate(timestamps), np.concatenate(cities_out))

def fit_feature_processors(x_train):
    n_features = x_train.shape[-1]
    flat_train = x_train.reshape(-1, n_features)
    imputer = SimpleImputer(strategy='median', add_indicator=False)
    scaler = StandardScaler()
    transformed = scaler.fit_transform(imputer.fit_transform(flat_train)).astype(np.float32)
    return imputer, scaler, transformed.reshape(x_train.shape)

def transform_features(x, imputer, scaler):
    shape = x.shape
    flat = x.reshape(-1, shape[-1])
    return scaler.transform(imputer.transform(flat)).astype(np.float32).reshape(shape)

def build_lstm(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(LOOKBACK_HOURS, n_features)),
        tf.keras.layers.LSTM(64, dropout=0.15, recurrent_dropout=0.0),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.10),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.Huber(), metrics=['mae'])
    return model

def metric_dict(y_true, y_pred):
    return {'mae_mw': mean_absolute_error(y_true, y_pred), 'rmse_mw': np.sqrt(mean_squared_error(y_true, y_pred)),
            'r2': r2_score(y_true, y_pred), 'nmae_percent_of_mean_load': 100 * mean_absolute_error(y_true, y_pred) / np.mean(y_true)}

def train_lstm(x_train, y_train, x_test, y_test, features):
    imputer, scaler, x_train_scaled = fit_feature_processors(x_train)
    x_test_scaled = transform_features(x_test, imputer, scaler)
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
    model = build_lstm(len(features))
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)]
    history = model.fit(x_train_scaled, y_train_scaled, validation_split=0.10, shuffle=False,
                        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1, callbacks=callbacks)
    pred_scaled = model.predict(x_test_scaled, batch_size=BATCH_SIZE, verbose=0).ravel()
    prediction = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    return model, imputer, scaler, y_scaler, history, prediction

## 1. Cross-city transfer LSTM

The held-out city is never used for fitting the imputer, input scaler, target scaler, early-stopping split, or network weights. No load-history columns enter these sequences.

In [5]:
transfer_results, transfer_predictions = [], []
for train_city, test_city in [('las_vegas', 'phoenix'), ('phoenix', 'las_vegas')]:
    tf.keras.backend.clear_session()
    x_train, y_train, _, _ = make_sequences(cities[train_city], TRANSFER_FEATURES)
    x_test, y_test, t_test, city_test = make_sequences(cities[test_city], TRANSFER_FEATURES)
    model, imputer, scaler, y_scaler, history, pred = train_lstm(x_train, y_train, x_test, y_test, TRANSFER_FEATURES)
    transfer_results.append({'experiment': 'cross_city_transfer_lstm', 'train_city': train_city, 'test_city': test_city, **metric_dict(y_test, pred)})
    pd.DataFrame({'timestamp': t_test, 'city': city_test, 'actual_mw': y_test, 'prediction_mw': pred, 'train_city': train_city, 'experiment': 'cross_city_transfer_lstm'}).to_csv(RESULTS_DIR / f'transfer_{train_city}_to_{test_city}_predictions.csv', index=False)
    model.save(RESULTS_DIR / f'transfer_{train_city}_to_{test_city}.keras')
    joblib.dump({'input_imputer': imputer, 'input_scaler': scaler, 'target_scaler': y_scaler, 'features': TRANSFER_FEATURES, 'lookback_hours': LOOKBACK_HOURS}, RESULTS_DIR / f'transfer_{train_city}_to_{test_city}_preprocessing.joblib')
    del x_train, x_test, model
    gc.collect()

transfer_results = pd.DataFrame(transfer_results)
display(transfer_results)

Epoch 1/60
247/247 [==============================] - 15s 16ms/step - loss: 0.1559 - mae: 0.4220 - val_loss: 0.4752 - val_mae: 0.8072
Epoch 2/60
247/247 [==============================] - 3s 12ms/step - loss: 0.1394 - mae: 0.3814 - val_loss: 0.2889 - val_mae: 0.5882
Epoch 3/60
247/247 [==============================] - 3s 13ms/step - loss: 0.1258 - mae: 0.3638 - val_loss: 0.2686 - val_mae: 0.5569
Epoch 4/60
247/247 [==============================] - 3s 13ms/step - loss: 0.1126 - mae: 0.3420 - val_loss: 0.2158 - val_mae: 0.4850
Epoch 5/60
247/247 [==============================] - 3s 12ms/step - loss: 0.1024 - mae: 0.3250 - val_loss: 0.1881 - val_mae: 0.4455
Epoch 6/60
247/247 [==============================] - 3s 12ms/step - loss: 0.0940 - mae: 0.3134 - val_loss: 0.1613 - val_mae: 0.4069
Epoch 7/60
247/247 [==============================] - 3s 12ms/step - loss: 0.0909 - mae: 0.3096 - val_loss: 0.1976 - val_mae: 0.4560
Epoch 8/60
247/247 [==============================] - 3s 12ms/step -

,experiment,train_city,test_city,mae_mw,rmse_mw,r2,nmae_percent_of_mean_load
0,cross_city_transfer_lstm,las_vegas,phoenix,2201.611572,2478.170595,-0.941522,38.340622
1,cross_city_transfer_lstm,phoenix,las_vegas,2521.498779,2578.219929,-9.175297,77.901121


## 2. Forward-in-time operational LSTM

Each city is trained on its first 80% of target timestamps and tested on its final 20%. Only the three approved past-load features are added; no current or future load is included.

In [6]:
operational_results = []
for city, df in cities.items():
    tf.keras.backend.clear_session()
    x_all, y_all, t_all, city_all = make_sequences(df, OPERATIONAL_FEATURES)
    cutoff = pd.Timestamp(df['timestamp'].iloc[int(len(df) * 0.80)])
    valid_load_lags = ~np.isnan(x_all[:, -1, [OPERATIONAL_FEATURES.index(f) for f in OPERATIONAL_LOAD_FEATURES]]).any(axis=1)
    train_mask = (pd.to_datetime(t_all) < cutoff) & valid_load_lags
    test_mask = (pd.to_datetime(t_all) >= cutoff) & valid_load_lags
    model, imputer, scaler, y_scaler, history, pred = train_lstm(x_all[train_mask], y_all[train_mask], x_all[test_mask], y_all[test_mask], OPERATIONAL_FEATURES)
    operational_results.append({'experiment': 'within_city_operational_lstm', 'train_city': city, 'test_city': city, 'cutoff': cutoff, **metric_dict(y_all[test_mask], pred)})
    pd.DataFrame({'timestamp': t_all[test_mask], 'city': city_all[test_mask], 'actual_mw': y_all[test_mask], 'prediction_mw': pred, 'train_city': city, 'experiment': 'within_city_operational_lstm'}).to_csv(RESULTS_DIR / f'operational_{city}_predictions.csv', index=False)
    model.save(RESULTS_DIR / f'operational_{city}.keras')
    joblib.dump({'input_imputer': imputer, 'input_scaler': scaler, 'target_scaler': y_scaler, 'features': OPERATIONAL_FEATURES, 'lookback_hours': LOOKBACK_HOURS}, RESULTS_DIR / f'operational_{city}_preprocessing.joblib')
    del x_all, model
    gc.collect()

operational_results = pd.DataFrame(operational_results)
display(operational_results)

Epoch 1/60
196/196 [==============================] - 5s 17ms/step - loss: 0.1197 - mae: 0.3653 - val_loss: 0.0478 - val_mae: 0.2332
Epoch 2/60
196/196 [==============================] - 3s 13ms/step - loss: 0.0669 - mae: 0.2782 - val_loss: 0.0274 - val_mae: 0.1886
Epoch 3/60
196/196 [==============================] - 2s 12ms/step - loss: 0.0456 - mae: 0.2285 - val_loss: 0.0196 - val_mae: 0.1581
Epoch 4/60
196/196 [==============================] - 2s 12ms/step - loss: 0.0391 - mae: 0.2061 - val_loss: 0.0151 - val_mae: 0.1355
Epoch 5/60
196/196 [==============================] - 2s 13ms/step - loss: 0.0520 - mae: 0.2343 - val_loss: 0.0140 - val_mae: 0.1328
Epoch 6/60
196/196 [==============================] - 3s 13ms/step - loss: 0.0319 - mae: 0.1834 - val_loss: 0.0168 - val_mae: 0.1457
Epoch 7/60
196/196 [==============================] - 2s 12ms/step - loss: 0.0353 - mae: 0.1901 - val_loss: 0.0240 - val_mae: 0.1731
Epoch 8/60
196/196 [==============================] - 2s 12ms/step - 

,experiment,train_city,test_city,cutoff,mae_mw,rmse_mw,r2,nmae_percent_of_mean_load
0,within_city_operational_lstm,las_vegas,las_vegas,2022-05-27,121.285225,178.143020,0.957200,3.486768
1,within_city_operational_lstm,phoenix,phoenix,2022-05-27,202.610962,287.930086,0.978351,3.184331


In [7]:
results = pd.concat([transfer_results, operational_results], ignore_index=True)
results.to_csv(RESULTS_DIR / 'lstm_model_metrics.csv', index=False)
with open(RESULTS_DIR / 'feature_sets.json', 'w', encoding='utf-8') as f:
    json.dump({'transfer_features': TRANSFER_FEATURES, 'operational_features': OPERATIONAL_FEATURES, 'lookback_hours': LOOKBACK_HOURS}, f, indent=2)
display(results)
print('Saved LSTM models, preprocessing objects, predictions, and metrics to:', RESULTS_DIR)

,experiment,train_city,test_city,mae_mw,rmse_mw,r2,nmae_percent_of_mean_load,cutoff
0,cross_city_transfer_lstm,las_vegas,phoenix,2201.611572,2478.170595,-0.941522,38.340622,NaT
1,cross_city_transfer_lstm,phoenix,las_vegas,2521.498779,2578.219929,-9.175297,77.901121,NaT
2,within_city_operational_lstm,las_vegas,las_vegas,121.285225,178.143020,0.957200,3.486768,2022-05-27
3,within_city_operational_lstm,phoenix,phoenix,202.610962,287.930086,0.978351,3.184331,2022-05-27


Saved LSTM models, preprocessing objects, predictions, and metrics to: D:\Downloads\BTP\model_training\lstm_training_3\lstm_training_results_cross_city_v1


## Deployment note

For a true day-ahead use case, replace each target-hour contemporaneous weather feature with its forecast counterpart before sequence construction. The transfer model measures weather-response portability; it cannot infer a new city's total system size without city metadata or a short calibration period.